# Streaming, Batch, and Async

Every LangChain **Runnable** exposes three execution modes beyond single **`invoke`**: **`stream`** for incremental output, **`batch`** for processing many inputs, and **async** variants (`ainvoke`, `astream`, `abatch`) for non-blocking servers. Because LCEL chains inherit the Runnable API, a composed pipeline `prompt | llm | parser` supports all modes without extra wiring.

This notebook covers when to use each mode, token streaming through chat models and chains, **`AIMessageChunk`** handling, **`astream_events`** for fine-grained traces, batch concurrency with **`max_concurrency`**, async patterns for FastAPI-style apps, streaming RAG chains, and production trade-offs (latency, cost, rate limits).

**02. Runnable Protocol and LCEL** introduced the API surface; **03. Chat Models and Messages** previewed model-level streaming. **06. LCEL Composition Patterns** used `batch` for map-reduce. Here we treat streaming and concurrency as first-class production concerns.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import asyncio
import time
from typing import Any

import langchain_core
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

print("langchain-core:", langchain_core.__version__)

---

## 1. Three Execution Modes — When to Use Each

| Mode | API | Best for |
|------|-----|----------|
| **Single sync** | `invoke(input)` | Simple scripts, synchronous API routes |
| **Streaming sync** | `stream(input)` | Chat UIs, SSE, time-to-first-token |
| **Batch sync** | `batch([inputs])` | Offline eval, bulk summarization, map steps |
| **Single async** | `await ainvoke(input)` | FastAPI, asyncio services |
| **Streaming async** | `async for chunk in astream(input)` | Async chat endpoints |
| **Batch async** | `await abatch([inputs])` | Concurrent I/O-bound bulk jobs |

```
invoke     ████████████████████  (wait for full response)
stream     ██░░░░░░░░░░░░░░░░░░  (first token early, then chunks)
batch      [job1][job2][job3]    (parallel where provider allows)
```

**Cost:** streaming does not reduce tokens billed — it changes **perceived latency**, not total generation cost.

---

## 2. Setup — Reusable Chain

We use a standard Q&A chain throughout this notebook:

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise assistant. Answer in 2-3 sentences."),
    ("human", "{question}"),
])

qa_chain = qa_prompt | llm | StrOutputParser()
print("chain ready:", type(qa_chain).__name__)

---

## 3. invoke — Baseline Latency

Measure full round-trip time before comparing streaming:

In [ ]:
question = {"question": "Explain JWT authentication briefly."}

t0 = time.perf_counter()
answer = qa_chain.invoke(question)
elapsed = time.perf_counter() - t0

print(f"invoke took {elapsed:.2f}s")
print("answer:", answer[:120], "..." if len(answer) > 120 else "")

---

## 4. stream — Token Chunks Through a Chain

`chain.stream(input)` yields **partial outputs** as the LLM generates. With **`StrOutputParser`**, chunks are **strings** (token text).

In [ ]:
print("Streaming tokens:")
chunks: list[str] = []
t_first = None
t0 = time.perf_counter()

for chunk in qa_chain.stream({"question": "What is Alembic used for?"}):
    if t_first is None:
        t_first = time.perf_counter() - t0
    chunks.append(chunk)
    print(chunk, end="", flush=True)

total = time.perf_counter() - t0
print(f"\n---\nchunks: {len(chunks)} | time-to-first-chunk: {t_first:.2f}s | total: {total:.2f}s")
print("reconstructed:", "".join(chunks)[:100], "...")

### 4.1 stream vs invoke — UX Trade-off

| Metric | `invoke` | `stream` |
|--------|----------|----------|
| Time to first visible token | After full completion | Much sooner |
| Total wall time | Similar | Similar |
| Client implementation | Simple | Iterator / SSE loop |
| Token cost | Same | Same |

Use **stream** when users watch text appear (chat UI). Use **invoke** for background jobs where no one waits interactively.

---

## 5. Model-Level Streaming — AIMessageChunk

Streaming **`llm`** directly (without `StrOutputParser`) yields **`AIMessageChunk`** objects:

In [ ]:
from langchain_core.messages import HumanMessage

print("llm.stream chunks (first 5 shown):")
for i, chunk in enumerate(llm.stream([HumanMessage(content="Say hello in 5 words.")])):
    if i < 5:
        print(f"  [{i}] type={type(chunk).__name__} content={chunk.content!r}")
    elif i == 5:
        print("  ...")
        break

# Reconstruct full text
full = "".join(c.content for c in llm.stream([HumanMessage(content="Say hello in 5 words.")]))
print("reconstructed:", full)

Chains with **`StrOutputParser`** abstract this — you receive plain strings. For metadata during streaming (tool calls mid-stream), stay at the model layer or use **`astream_events`** (§9).

---

## 6. astream — Async Streaming

Use **`async for`** in FastAPI and asyncio applications:

In [ ]:
async def stream_demo():
    print("astream:", end=" ")
    async for chunk in qa_chain.astream({"question": "What is pytest?"}):
        print(chunk, end="", flush=True)
    print()


asyncio.run(stream_demo())

### 6.1 FastAPI SSE Pattern (Sketch)

Typical integration wraps `astream` in a **`StreamingResponse`**:

In [ ]:
# Conceptual FastAPI handler — not a running server
FASTAPI_SSE_EXAMPLE = '''
from fastapi import FastAPI
from fastapi.responses import StreamingResponse

app = FastAPI()

@app.post("/chat/stream")
async def chat_stream(question: str):
    async def token_generator():
        async for chunk in qa_chain.astream({"question": question}):
            yield chunk  # or format as SSE: f"data: {chunk}\\n\\n"
    return StreamingResponse(token_generator(), media_type="text/plain")
'''
print(FASTAPI_SSE_EXAMPLE)

Production APIs add auth, rate limits, and error boundaries — patterns in **17. Production Patterns for LangChain**.

---

## 7. batch — Parallel Inputs

`batch` accepts a **list of inputs** and returns a **list of outputs** in the same order.

In [ ]:
questions = [
    {"question": "What is HTTP?"},
    {"question": "What is REST?"},
    {"question": "What is OpenAPI?"},
]

t0 = time.perf_counter()
answers = qa_chain.batch(questions)
elapsed = time.perf_counter() - t0

print(f"batch({len(questions)}) took {elapsed:.2f}s\n")
for q, a in zip(questions, answers):
    print(f"Q: {q['question']}")
    print(f"A: {a}\n")

### 7.1 max_concurrency — Rate Limit Control

Pass **`RunnableConfig(max_concurrency=N)`** to cap parallel requests:

In [ ]:
from langchain_core.runnables import RunnableConfig

many = [{"question": f"Define term {i} in one word."} for i in range(6)]

cfg_serial = RunnableConfig(max_concurrency=1)
cfg_parallel = RunnableConfig(max_concurrency=4)

t0 = time.perf_counter()
qa_chain.batch(many, config=cfg_serial)
serial_time = time.perf_counter() - t0

t0 = time.perf_counter()
qa_chain.batch(many, config=cfg_parallel)
parallel_time = time.perf_counter() - t0

print(f"max_concurrency=1: {serial_time:.2f}s")
print(f"max_concurrency=4: {parallel_time:.2f}s")

Tune **`max_concurrency`** against provider rate limits (RPM/TPM). Too high triggers 429 errors; too low underutilizes throughput.

---

## 8. Async Batch and Invoke

### 8.1 ainvoke

In [ ]:
async def ainvoke_demo():
    result = await qa_chain.ainvoke({"question": "What is asyncio?"})
    print("ainvoke:", result)


asyncio.run(ainvoke_demo())

### 8.2 abatch

In [ ]:
async def abatch_demo():
    inputs = [
        {"question": "What is a coroutine?"},
        {"question": "What is an event loop?"},
    ]
    results = await qa_chain.abatch(inputs)
    for r in results:
        print(" -", r[:60], "..." if len(r) > 60 else "")


asyncio.run(abatch_demo())

### 8.3 asyncio.gather — Multiple Independent Chains

When chains are unrelated, **`gather`** overlaps waiting time:

In [ ]:
async def gather_demo():
    tasks = [
        qa_chain.ainvoke({"question": "What is JWT?"}),
        qa_chain.ainvoke({"question": "What is OAuth2?"}),
        qa_chain.ainvoke({"question": "What is API key auth?"}),
    ]
    t0 = time.perf_counter()
    results = await asyncio.gather(*tasks)
    print(f"gather 3 calls: {time.perf_counter() - t0:.2f}s")
    for r in results:
        print(" -", r[:50], "...")


asyncio.run(gather_demo())

---

## 9. astream_events — Fine-Grained Streaming

**`astream_events`** emits structured events for each step in a chain — useful for debugging, progress UI, and logging intermediate states.

In [ ]:
async def events_demo():
    print("astream_events (selected):")
    async for event in qa_chain.astream_events(
        {"question": "What is FastAPI?"},
        version="v2",
    ):
        kind = event.get("event")
        if kind in ("on_chat_model_stream", "on_chain_end"):
            name = event.get("name", "")
            if kind == "on_chat_model_stream":
                chunk = event.get("data", {}).get("chunk")
                if chunk and getattr(chunk, "content", None):
                    print(chunk.content, end="", flush=True)
            elif name == "StrOutputParser":
                print(f"\n[chain_end: {name}]")
    print()


asyncio.run(events_demo())

| Event (examples) | Meaning |
|------------------|--------|
| `on_chain_start` / `on_chain_end` | Step entered / finished |
| `on_chat_model_stream` | Token chunk from LLM |
| `on_parser_stream` | Parser emitted partial output |

Use **`version="v2"`** for the current event schema. Deeper observability hooks are in **15. Callbacks, Caching, and Observability**.

---

## 10. Streaming RAG Chains

RAG compositions (**06**, **11**) stream the **final answer tokens** the same way — retrieval runs before generation (non-streaming retrieval step, streaming LLM step):

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

CORPUS = {
    "jwt": "JWT bearer tokens use Authorization: Bearer header.",
    "alembic": "Run alembic upgrade head to apply migrations.",
}


def fake_retrieve(q: str) -> str:
    ql = q.lower()
    if "jwt" in ql or "auth" in ql:
        return CORPUS["jwt"]
    return CORPUS.get("alembic", "No docs.")


rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using context only.\n{context}"),
    ("human", "{question}"),
])

rag_chain = (
    RunnableParallel(context=RunnableLambda(fake_retrieve), question=RunnablePassthrough())
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("Streaming RAG answer:")
for chunk in rag_chain.stream("What header carries JWT?"):
    print(chunk, end="", flush=True)
print("\n--- done ---")

**Note:** retrieval completes before the first token streams. For long retrievers, show a "Searching…" UI before streaming starts.

---

## 11. Streaming Structured Output

**JSON/Pydantic parsers** (**05. Output Parsers and Structured Output**) typically need the **full** completion before parsing. Options for "streaming structured data":

| Approach | Streaming? |
|----------|------------|
| `StrOutputParser` + prose | Yes — token chunks |
| `JsonOutputParser` at chain end | No — parse after full JSON |
| Provider structured output API | Provider-dependent partial JSON |
| Stream prose, parse post-hoc | Stream UX, validate at end |

Default: stream **text** for UX; validate **structure** after the stream completes.

---

## 12. batch for Map-Reduce and Eval

**06. LCEL Composition Patterns** map-reduce uses **`batch`** on the map step. Same pattern for golden-set evaluation:

In [ ]:
GOLDEN = [
    {"question": "What is JWT?", "expect_keyword": "Bearer"},
    {"question": "Alembic upgrade command?", "expect_keyword": "upgrade"},
    {"question": "What is HTTP?", "expect_keyword": "protocol"},
]

eval_inputs = [{"question": g["question"]} for g in GOLDEN]
eval_outputs = qa_chain.batch(eval_inputs, config=RunnableConfig(max_concurrency=2))

print(f"{'question':35s} {'pass':5s} snippet")
print("-" * 70)
for g, out in zip(GOLDEN, eval_outputs):
    passed = g["expect_keyword"].lower() in out.lower()
    print(f"{g['question'][:35]:35s} {str(passed):5s} {out[:40]}...")

---

## 13. Sync vs Async in Production

| Context | Recommendation |
|---------|----------------|
| **FastAPI / Starlette** | `ainvoke`, `astream` on async routes |
| **Flask / sync WSGI** | `invoke`, `stream` or run async in thread pool |
| **Jupyter notebook** | `invoke`, `stream`; `asyncio.run()` for async demos |
| **Batch CLI scripts** | `batch` with tuned `max_concurrency` |
| **Celery / RQ workers** | `invoke` per task; worker concurrency handles parallelism |

Do not call **`invoke`** inside async route handlers without **`asyncio.to_thread`** — it blocks the event loop.

In [ ]:
# Illustrative latency comparison (values vary by network/load)
modes = ["invoke", "stream (TTFB)", "batch x3"]
latency = [2.1, 0.4, 2.8]  # illustrative seconds

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(modes, latency, color=["#4c72b0", "#55a868", "#c44e52"])
ax.set_ylabel("seconds (illustrative)")
ax.set_title("Streaming improves time-to-first-chunk, not always total batch time")
plt.tight_layout()
plt.show()

---

## 14. RunnableLambda — Async Functions

Async lambdas integrate with **`ainvoke`** / **`astream`** automatically (**02. Runnable Protocol and LCEL**):

In [ ]:
from langchain_core.runnables import RunnableLambda


async def async_upper(text: str) -> str:
    await asyncio.sleep(0.01)
    return text.upper()


async_chain = RunnableLambda(async_upper) | qa_prompt | llm | StrOutputParser()


async def run_async_chain():
    result = await async_chain.ainvoke({"question": "what is lcel?"})
    print("async chain:", result[:80])


asyncio.run(run_async_chain())

---

## 15. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| `invoke` in async route | Blocked event loop | Use `ainvoke` |
| Expect JSON parser to stream | Partial invalid JSON | Stream text; parse at end |
| `batch` without concurrency limit | 429 rate errors | Set `max_concurrency` |
| Ignoring retrieval latency in RAG stream | Frozen UI before tokens | Show loading state |
| Concatenating chunks incorrectly | Broken Unicode | Join strings in order |
| Assuming batch order differs | Wrong result mapping | `batch` preserves input order |
| Double API calls in stream demo | 2× cost | Reuse one stream iterator |

---

## 16. Mode Selection Cheat Sheet

| Goal | API |
|------|-----|
| One answer, sync | `chain.invoke(x)` |
| Chat UI tokens | `for c in chain.stream(x)` |
| FastAPI streaming | `async for c in chain.astream(x)` |
| Many prompts, sync | `chain.batch(list, config=...)` |
| Many prompts, async | `await chain.abatch(list)` |
| Debug step events | `async for e in chain.astream_events(x, version="v2")` |
| Overlap unrelated calls | `await asyncio.gather(...)` |

---

## 17. Summary

LangChain chains inherit **invoke**, **stream**, **batch**, and **async** variants from the Runnable protocol. **Streaming** improves time-to-first-token for chat UX; **batch** parallelizes independent inputs with **`max_concurrency`**; **async** methods integrate with modern Python web servers.

Key takeaways:

- **`chain.stream`** + **`StrOutputParser`** yields user-visible text chunks.
- **`llm.stream`** yields **`AIMessageChunk`** — use when you need raw model chunks.
- **`astream_events(version="v2")`** exposes per-step events for debugging and rich UI.
- **`batch` / `abatch`** preserve order; tune **`max_concurrency`** for rate limits.
- RAG streams **generation** after **retrieval** completes.
- Structured parsers generally require full output — stream prose, parse after.

Demonstrations covered invoke vs stream timing, model chunks, async stream/batch/gather, events API, streaming RAG, eval batch, and async lambda chains.

Next: **08. Documents and Text Splitting** — `Document` objects, chunking strategies, and preparing text for retrieval.